# 48 — AR Collection Agent

**Pattern:** Aging-bucket state machine + tone-calibrated letter generation

This workbook walks through how deterministic business rules (bucket assignment,
priority scoring, credit-hold logic) combine with selective LLM invocation
(one SystemMessage per escalation tier) to produce an auditable AR collection plan.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain-openai langchain-core pydantic python-dotenv

## 2. Set API Key

In [ ]:
import os

# Paste your OpenAI API key here for Colab, or load from .env locally
os.environ["OPENAI_API_KEY"] = "sk-..."  # replace with your key

## Part 1 — Business Problem

Finance teams chasing overdue invoices face a tone problem:

- A 12-day overdue invoice deserves a **warm, collaborative reminder** — the customer
  likely just forgot.
- A 75-day overdue invoice needs a **firm, formal notice** with clear deadlines.
- A 95-day overdue invoice requires a **final demand** with explicit legal warnings.
- Accounts over 90 days with high credit exposure need an **internal legal referral memo**.

Writing separate, appropriately-toned letters for each tier manually is slow and inconsistent.
This agent automates the classification (deterministic) and the letter drafting (LLM),
while keeping the business rules auditable.

## Part 2 — Schema

Three Pydantic models define the data flow:

- **ARCustomer** — input: customer and invoice details
- **CollectionAction** — output per customer: bucket, tier, letter, score, hold
- **CollectionPlan** — aggregate output: all actions + plan totals

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field


class ARCustomer(BaseModel):
    customer_id: str = Field(description="Unique customer identifier.")
    customer_name: str = Field(description="Legal name of the customer entity.")
    invoice_number: str = Field(description="Invoice reference number.")
    invoice_date: str = Field(description="Invoice issue date (YYYY-MM-DD).")
    due_date: str = Field(description="Payment due date (YYYY-MM-DD).")
    outstanding_amount: float = Field(description="Amount outstanding on this invoice.")
    currency: str = Field(default="USD", description="Currency code.")
    days_overdue: int = Field(description="Calendar days past the due date.")
    prior_contact_count: int = Field(default=0, description="Prior collection contact attempts.")
    credit_limit: float = Field(description="Approved credit limit for this customer.")
    total_exposure: float = Field(description="Total open balance across all invoices.")


class CollectionAction(BaseModel):
    customer_id: str
    aging_bucket: Literal["current", "1_30", "31_60", "61_90", "90_plus"]
    escalation_tier: Literal[
        "no_action", "friendly_reminder", "formal_notice", "final_demand", "legal_referral"
    ]
    collection_letter: str
    credit_hold_recommended: bool
    credit_hold_rationale: Optional[str] = None
    priority_score: int  # 1-10


class CollectionPlan(BaseModel):
    as_of_date: str
    total_ar_outstanding: float
    actions: List[CollectionAction]  # sorted by priority_score desc
    credit_hold_count: int
    legal_referral_count: int
    collection_summary: str


print("Schema defined: ARCustomer, CollectionAction, CollectionPlan")

## Part 3 — Escalation Tiers and Tone Calibration

### BUCKET_MAPPING — deterministic, no LLM involved

The aging bucket is computed directly from `days_overdue`, then the tier is
looked up in a dict. This is intentionally not an LLM decision — it must be
consistent and auditable.

```
days_overdue=0        -> current   -> no_action
days_overdue=1..30    -> 1_30      -> friendly_reminder
days_overdue=31..60   -> 31_60     -> formal_notice
days_overdue=61..90   -> 61_90     -> final_demand
days_overdue=91+      -> 90_plus   -> legal_referral
```

### ESCALATION_PROMPTS — why they differ in tone

Each tier's SystemMessage is tuned to a specific persona and word budget:

| Tier | Persona | Tone | ~Words | Key Requirement |
|:---|:---|:---|:---:|:---|
| friendly_reminder | AR Specialist | Warm, collaborative | 150 | Good-standing ack, payment methods |
| formal_notice | AR Manager | Firm, professional | 200 | 7-day deadline, consequences |
| final_demand | AR Director | Serious, unambiguous | 250 | 5-business-day deadline, legal warning |
| legal_referral | AR → Legal | Internal memo | 200 | Memo format, recommended action |

The `legal_referral` prompt is qualitatively different: it is NOT a customer-facing
letter — it is an internal memo addressed **TO: Legal Department**, summarising the
case for handoff.

In [ ]:
from langchain_core.messages import SystemMessage

BUCKET_MAPPING = {
    "current": "no_action",
    "1_30": "friendly_reminder",
    "31_60": "formal_notice",
    "61_90": "final_demand",
    "90_plus": "legal_referral",
}

ESCALATION_PROMPTS = {
    "friendly_reminder": SystemMessage(
        content=(
            "You are a polite and professional accounts receivable specialist. "
            "Write a warm, friendly payment reminder letter of approximately 150 words.\n\n"
            "Tone: Assume the overdue payment is simply an oversight. "
            "Required: acknowledge good standing, state invoice and amount, "
            "request payment, list payment methods."
        )
    ),
    "formal_notice": SystemMessage(
        content=(
            "You are a firm but professional accounts receivable manager. "
            "Write a formal payment notice letter of approximately 200 words.\n\n"
            "Required: invoice number and amount, 7-day payment deadline, "
            "consequences (interest, credit hold), payment instructions."
        )
    ),
    "final_demand": SystemMessage(
        content=(
            "You are a senior accounts receivable director. "
            "Write a serious, unambiguous final demand letter of approximately 250 words.\n\n"
            "Required: account history summary, 5-business-day deadline, "
            "explicit legal warning (referral to counsel + potential litigation), "
            "final payment instructions."
        )
    ),
    "legal_referral": SystemMessage(
        content=(
            "You are an accounts receivable manager preparing an internal legal referral memo. "
            "Write a concise internal memo of approximately 200 words.\n\n"
            "Format: TO: Legal Department / FROM: Accounts Receivable / "
            "RE: Customer [Name] — Overdue Account Referral\n\n"
            "Required: customer ID, total exposure, days overdue, "
            "collection attempt summary, recommended action."
        )
    ),
}

print("BUCKET_MAPPING:", BUCKET_MAPPING)
print("\nTiers with ESCALATION_PROMPTS:", list(ESCALATION_PROMPTS.keys()))

## Part 4 — Workflow Functions

Three deterministic helpers + one orchestration function:

In [ ]:
from langchain_openai import ChatOpenAI


def _get_bucket(days_overdue: int) -> str:
    """Deterministically map days_overdue to an aging bucket."""
    if days_overdue <= 0:
        return "current"
    if days_overdue <= 30:
        return "1_30"
    if days_overdue <= 60:
        return "31_60"
    if days_overdue <= 90:
        return "61_90"
    return "90_plus"


def _priority_score(customer: ARCustomer) -> int:
    """Score from 1 (low) to 10 (high) based on days overdue and amount."""
    raw = customer.days_overdue / 10 + customer.outstanding_amount / 5000
    return min(10, max(1, int(raw)))


def _credit_hold(customer: ARCustomer):
    """Recommend credit hold if severely overdue or high exposure ratio."""
    reasons = []
    if customer.days_overdue > 90:
        reasons.append(f"{customer.days_overdue} days overdue (threshold: 90)")
    ratio = customer.total_exposure / customer.credit_limit if customer.credit_limit > 0 else 0
    if ratio > 0.8:
        reasons.append(f"exposure {ratio*100:.1f}% of credit limit")
    if reasons:
        return True, "Credit hold recommended: " + "; ".join(reasons) + "."
    return False, None


# Verify bucket mapping
test_cases = [0, 15, 45, 75, 100]
for d in test_cases:
    b = _get_bucket(d)
    t = BUCKET_MAPPING[b]
    print(f"days_overdue={d:3d}  ->  bucket={b:<8}  ->  tier={t}")

## Part 5 — Mini Run: 2 Customers

Run BrightPath Ltd (friendly_reminder) and Delta Enterprises (final_demand)
to see the tone contrast between two tiers.

In [ ]:
def run_mini(customers, as_of_date="2025-06-24", model="gpt-4.1-nano"):
    llm = ChatOpenAI(model=model, temperature=0.3, api_key=os.environ["OPENAI_API_KEY"])
    actions = []
    for customer in customers:
        bucket = _get_bucket(customer.days_overdue)
        tier = BUCKET_MAPPING[bucket]
        score = _priority_score(customer)
        hold, rationale = _credit_hold(customer)
        if tier == "no_action":
            letter = "No action required."
        else:
            sys_msg = ESCALATION_PROMPTS[tier]
            user_content = (
                f"Customer: {customer.customer_name} (ID: {customer.customer_id})\n"
                f"Invoice: {customer.invoice_number}\n"
                f"Due date: {customer.due_date}\n"
                f"Outstanding: {customer.currency} {customer.outstanding_amount:,.2f}\n"
                f"Days overdue: {customer.days_overdue}\n"
                f"Prior contacts: {customer.prior_contact_count}\n"
                f"Total exposure: {customer.currency} {customer.total_exposure:,.2f}"
            )
            chain = sys_msg | llm
            resp = chain.invoke({"messages": [{"role": "user", "content": user_content}]})
            letter = resp.content
        actions.append(
            CollectionAction(
                customer_id=customer.customer_id,
                aging_bucket=bucket,
                escalation_tier=tier,
                collection_letter=letter,
                credit_hold_recommended=hold,
                credit_hold_rationale=rationale,
                priority_score=score,
            )
        )
    return actions


mini_customers = [
    ARCustomer(
        customer_id="CUST-002",
        customer_name="BrightPath Ltd",
        invoice_number="INV-2025-0388",
        invoice_date="2025-05-17",
        due_date="2025-06-06",
        outstanding_amount=12400.00,
        days_overdue=18,
        prior_contact_count=0,
        credit_limit=50000.00,
        total_exposure=12400.00,
    ),
    ARCustomer(
        customer_id="CUST-004",
        customer_name="Delta Enterprises",
        invoice_number="INV-2025-0244",
        invoice_date="2025-03-15",
        due_date="2025-04-14",
        outstanding_amount=22000.00,
        days_overdue=72,
        prior_contact_count=2,
        credit_limit=50000.00,
        total_exposure=22000.00,
    ),
]

mini_actions = run_mini(mini_customers)

for action in mini_actions:
    cust = next(c for c in mini_customers if c.customer_id == action.customer_id)
    print(f"\n{'='*60}")
    print(f"Customer : {cust.customer_name}")
    print(f"Tier     : {action.escalation_tier}")
    print(f"Priority : {action.priority_score}")
    print(f"Hold     : {action.credit_hold_recommended}")
    print(f"\nLetter:\n{action.collection_letter}")

## Part 6 — Full 6-Customer Plan

Run all 6 customers across all 5 aging buckets and print the complete plan.

In [ ]:
all_customers = [
    ARCustomer(
        customer_id="CUST-001", customer_name="Apex Corp",
        invoice_number="INV-2025-0441", invoice_date="2025-06-10", due_date="2025-06-24",
        outstanding_amount=8750.00, days_overdue=0, prior_contact_count=0,
        credit_limit=50000.00, total_exposure=8750.00,
    ),
    ARCustomer(
        customer_id="CUST-002", customer_name="BrightPath Ltd",
        invoice_number="INV-2025-0388", invoice_date="2025-05-17", due_date="2025-06-06",
        outstanding_amount=12400.00, days_overdue=18, prior_contact_count=0,
        credit_limit=50000.00, total_exposure=12400.00,
    ),
    ARCustomer(
        customer_id="CUST-003", customer_name="Coastal Trading",
        invoice_number="INV-2025-0312", invoice_date="2025-04-20", due_date="2025-05-10",
        outstanding_amount=31500.00, days_overdue=45, prior_contact_count=1,
        credit_limit=50000.00, total_exposure=31500.00,
    ),
    ARCustomer(
        customer_id="CUST-004", customer_name="Delta Enterprises",
        invoice_number="INV-2025-0244", invoice_date="2025-03-15", due_date="2025-04-14",
        outstanding_amount=22000.00, days_overdue=72, prior_contact_count=2,
        credit_limit=50000.00, total_exposure=22000.00,
    ),
    ARCustomer(
        customer_id="CUST-005", customer_name="Echo Systems",
        invoice_number="INV-2025-0180", invoice_date="2025-02-10", due_date="2025-03-12",
        outstanding_amount=85000.00, days_overdue=97, prior_contact_count=3,
        credit_limit=100000.00, total_exposure=120000.00,
    ),
    ARCustomer(
        customer_id="CUST-006", customer_name="Frontier Corp",
        invoice_number="INV-2025-0201", invoice_date="2025-04-25", due_date="2025-05-20",
        outstanding_amount=4200.00, days_overdue=35, prior_contact_count=0,
        credit_limit=50000.00, total_exposure=4200.00,
    ),
]

full_actions = run_mini(all_customers, as_of_date="2025-06-24")
full_actions.sort(key=lambda a: a.priority_score, reverse=True)

total_ar = sum(c.outstanding_amount for c in all_customers)
hold_count = sum(1 for a in full_actions if a.credit_hold_recommended)
legal_count = sum(1 for a in full_actions if a.escalation_tier == "legal_referral")

print(f"Total AR outstanding : ${total_ar:,.2f}")
print(f"Credit holds         : {hold_count}")
print(f"Legal referrals      : {legal_count}")
print()
print(f"{'Customer':<22} {'Days OD':>8} {'Tier':<20} {'Score':>6} {'Hold':>6}")
print("-" * 68)
for action in full_actions:
    cust = next(c for c in all_customers if c.customer_id == action.customer_id)
    hold_flag = "YES" if action.credit_hold_recommended else "no"
    print(
        f"{cust.customer_name:<22} {cust.days_overdue:>8} "
        f"{action.escalation_tier:<20} {action.priority_score:>6} {hold_flag:>6}"
    )

## Part 7 — Exercise

### Challenge: Add a `pre_legal` Tier

The current ladder jumps from `final_demand` (61-90 days) straight to `legal_referral` (91+ days).
Product wants a new intermediate tier for accounts that are **75-89 days overdue**:
a stern-but-not-final-demand tone that makes clear legal action is imminent but
still gives the customer a chance to resolve.

**What changes are required?**

1. **`prompts.py`** — what do you add?
2. **`_get_bucket`** — what logic changes?
3. **`schema.py`** — what do you update?

Try to write the changes before looking at the answer key below.

In [ ]:
# Your solution here

# 1. New entry in BUCKET_MAPPING:
# BUCKET_MAPPING["61_74"] = "final_demand"
# BUCKET_MAPPING["75_89"] = "pre_legal"
# BUCKET_MAPPING["90_plus"] = "legal_referral"

# 2. New SystemMessage in ESCALATION_PROMPTS:
# ESCALATION_PROMPTS["pre_legal"] = SystemMessage(content=...)

# 3. Updated Literal in CollectionAction.aging_bucket and .escalation_tier

# 4. Updated _get_bucket thresholds
pass

### Answer Key

**1. `prompts.py` — add to BUCKET_MAPPING and ESCALATION_PROMPTS**

```python
# BUCKET_MAPPING additions
BUCKET_MAPPING = {
    "current": "no_action",
    "1_30": "friendly_reminder",
    "31_60": "formal_notice",
    "61_74": "final_demand",       # renamed upper bound
    "75_89": "pre_legal",           # new bucket
    "90_plus": "legal_referral",
}

# New SystemMessage for pre_legal
ESCALATION_PROMPTS["pre_legal"] = SystemMessage(
    content=(
        "You are a senior accounts receivable manager writing a pre-legal warning letter. "
        "Tone: Stern but not a final demand — make it unambiguous that legal referral is "
        "imminent (within 10 business days) unless payment is received, but still leave "
        "the door open for the customer to resolve the matter directly. ~220 words. "
        "Required: invoice number, amount, days overdue, prior contact count, "
        "10-business-day deadline, explicit statement that legal referral is being prepared."
    )
)
```

**2. `_get_bucket` — split the 61-90 day range**

```python
def _get_bucket(days_overdue: int) -> str:
    if days_overdue <= 0:
        return "current"
    if days_overdue <= 30:
        return "1_30"
    if days_overdue <= 60:
        return "31_60"
    if days_overdue <= 74:
        return "61_74"     # was 61_90
    if days_overdue <= 89:
        return "75_89"     # new bucket
    return "90_plus"
```

**3. `schema.py` — update Literal fields**

```python
aging_bucket: Literal["current", "1_30", "31_60", "61_74", "75_89", "90_plus"]

escalation_tier: Literal[
    "no_action", "friendly_reminder", "formal_notice",
    "final_demand", "pre_legal", "legal_referral"
]
```

**Key insight:** The only place the new tier *feels* different from `final_demand` is
the `SystemMessage` — the state machine structure (dict lookup, bucket function,
schema literal) is boilerplate. This is the core advantage of the pattern:
adding a new tier is a localised, low-risk change.